# Pre-processing of the EDGAR 10-k filings              


Note: The code for pre-processing the 10-K filings is documented in this file, as for some steps a GPU is required and, hence, the requirements/packages are slightly different (see requirement.txt file). However, if a sufficiently powerful (NVIDIA) GPU is not available, the code can also be run on a CPU. In this case, a different (CPU-based) model version is reffered to in the code below. The CPU-based model is sometimes less precise, but it is still sufficient for the presented purposes. BUT: The overall results may differ slightly, as the CPU-based model is not exactly the same as the GPU-based model.

<div class="alert-warning">
Libraries
</div>

First, we Import all necessary libraries. These inlcude "os" and "pathlib" to set and handle working directories, "pandas" and "numpy" for data handling and calculations, "pickle" to save the prepared data as memory efficient pickle files,"re" to handle text/string data, and "spacy" for NLP tasks. 

In [ ]:
# If you want to use the GPU-based version of Spacy, you need to install the following in the right order:
# 1. Install the GPU driver for your graphics card (if you haven't already). 

# 2. Install the CUDA toolkit that is compatible with your GPU and the version of PyTorch you want to use. 
# You can check the compatibility of your GPU with the CUDA toolkit on the NVIDIA website: https://developer.nvidia.com/cuda-toolkit.

# 3. Install PyTorch with GPU support using the appropriate command. You need to install it and make sure it is compatible with your GPU. 
# You can check the compatibility of your GPU with the torch library on the PyTorch website: https://pytorch.org/get-started/locally/
# Make sure to install the correct version of torch that is compatible with your GPU and CUDA version.

# 4. Install the GPU-based version of Spacy using the command:
# pip install spacy[cuda]

import os
from pathlib import Path
import pandas as pd
import numpy as np
import re
import pickle
from tqdm import tqdm
import torch
import spacy
import itertools
import collections
import openpyxl
from nltk.corpus import stopwords

from gensim.models.phrases import Phrases, Phraser, ENGLISH_CONNECTOR_WORDS 
from gensim.models import word2vec, Word2Vec

<div class="alert-warning">
Check wether GPU is available for NLP tasks (just necessary for W2V pre-processing)
</div>

In [ ]:
# Check if spaCy is using GPU
is_gpu_enabled = spacy.prefer_gpu()

print(f"Is GPU enabled: {is_gpu_enabled}")

<div class="alert-warning">
Set the working directory
</div>

In [ ]:
os.chdir('../../data')

<div class="alert-info">
Step 1: Define further functions needed for pre-processing 
</div>

In the following we will define some functions that are helpful to further process the data.

In [ ]:
def remove_numbers(text):
    """
    Function to replace numbers by  " "
    """
    text = re.sub(r'\b\d+(?:[.,]\d{3})*(?:[.,]\d+)?\b', ' ',text)
    return text

def remove_punctuation_and_nonalpha(text):
    """
    Function to remove the punctuation, upper casing and words that include
    non-alpha characters.
    """
    chars = '.,\'´-!$%&()'
    text = text.translate(str.maketrans(chars, ' ' * len(chars)))
    text = re.sub(r'\s{2,}', ' ', text.strip())
    return ' '.join([word.lower() for word in text.split() if word.isalpha()])

def remove_stopwords(text, stop_words):
    """ Function to remove stopwords. """
    return ' '.join([word for word in str(text).split() if word not in stop_words])

def remove_short_long_words(text):
    """ Function to remove too long and too short words. """
    return ' '.join([word for word in str(text).split() if len(word) >= 2 and len(word) < 16])

def split_into_sentences(doc):
    """ Splits the document into sentences using spaCy.
    """
    return [sent.text for sent in doc.sents]


def count_words_in_short_sentences(texts):
    """
    Counts the number of words in sentences shorter than 6 words.
    """
    return sum(len(re.findall(r'\b[a-zA-Z]+\b', text)) for text in texts if len(re.findall(r'\b[a-zA-Z]+\b', text)) < 6)

def count_words_in_long_sentences(texts):
    """
    Counts the number of words in sentences longer than 800 words.
    """
    return sum(len(re.findall(r'\b[a-zA-Z]+\b', text)) for text in texts if len(re.findall(r'\b[a-zA-Z]+\b', text)) > 800)


<div class="alert-info">
Step 2.1: Pre-processing of 10-K filings for a traditional dictionary (i.e., word representation as term-matrix) approach (This holds for the Bag-of-Word (BoW)-based and Word-2-Vec (W2V)-based dirctionaries )
</div>

To use the 10-K filing text in a basic BoW approach we remove numbers, punctuation, upper casing, words that include non-alphabetic characters (i.e., any remaining symbol that is not part of the alphabet), stopwords provided by NLTK package (e.g., "a", "the", and "in"), and too short (<= 2 characters) or too long words (> 16 characters), and any parts-of-speech except nouns, verbs, adverbs, and adjectives (this is done via a deep learning model provided by spaCy).

In [ ]:
# Load the corpus data frame from the pickle file

with open('BoW/Intermediate_datasets/Corpus_df_HTML_cleaned_BoW_v1.pkl','rb') as path_name:
    corpus_df = pickle.load(path_name)

stop_words = stopwords.words("english")

#You can add some custom stopwords (you can apply all the functions for lists)
#stop_words.append('some_word_you_dont_like')  

#Loop through the filings and further pre-process the text
for doc in range(0,len(corpus_df["filing_text"])):

    #Let the user know how many documents are pre-processed (in steps of 50)
    if doc%50 == 0:
        print(f"{doc} documents have been processed")
    
    #Step 1: Remove number 
    text = remove_numbers(corpus_df.loc[doc,"filing_text"])

    #Step 2: Remove punctuation, upper casing, words that include non-alpha characters
    text = remove_punctuation_and_nonalpha(text)

    #Step 3: Remove predefined stopwords
    text = remove_stopwords(text, stop_words)

    #Step 4: Remove too short (<= 2 characters) and too long words (>16 characters)
    text = remove_short_long_words(text)

    corpus_df.loc[doc,"filing_text"] = text


#Save the pre-processed corpus as pickle format
corpus_df.to_pickle("BoW/Intermediate_datasets/Corpus_df_HTML_cleaned_BoW_final.pkl")

In addition, we prepare the corpus for the W2V-based dictionary in a similar way. Note that the pre-processing for the W2V model itself is different (see below).

In [ ]:
# Load the corpus data frame from the pickle file
with open('BoW/Intermediate_datasets/Corpus_df_HTML_cleaned_BoW_v1.pkl','rb') as path_name:
    corpus_df = pickle.load(path_name)

# Some stopwords should not be removed in the case of the W2V approach (this depends on the dictionary). 
# Hence, we remove some specified words from the stop_words list
words_to_remove = {'it', 'of', 'in', 'on', 'at', 'to', 'for', 'our', 'after'}

stop_words = [word for word in stop_words if word not in words_to_remove]

#Loop through the filings and further pre-process the text
for doc in range(0,len(corpus_df["filing_text"])):

    #Let the user know how many documents are pre-processed (in steps of 50)
    if doc%50 == 0:
        print(f"{doc} documents have been processed")
    
    #Step 1: Remove number 
    text = remove_numbers(corpus_df.loc[doc,"filing_text"])

    #Step 2: Remove punctuation, upper casing, words that include non-alpha characters
    text = remove_punctuation_and_nonalpha(text)

    #Step 3: Remove predefined stopwords
    text = remove_stopwords(text, stop_words)

    corpus_df.loc[doc,"filing_text"] = text

# Save the further cleaned BoW corpus

if not os.path.exists('W2V/Intermediate_datasets'):
    os.makedirs('W2V/Intermediate_datasets')

with open('W2V/Intermediate_datasets/Corpus_df_HTML_cleaned_W2V_final.pkl','wb') as path_name:
    pickle.dump(corpus_df, path_name)

<div class="alert-info">
Step 2.2: Pre-processing of 10-K filings for a Word-embedding approach (Word-2-Vec or GLLM)
</div>

To use the 10-K filing text for advanced word embedding models, nothing is removed in order to maintain the thread of the text. The upcoming clean-up steps are split in two parts. The first part is the same of the Word-2-Vec and GLLM aprroaches and the second part just applies for the Word-2-Vec approach.

<div class="alert-info">
Part 1 (Same for Word-2-Vec and GLLM)
</div>

First, we load the parsed corpus and then split the filing into a list of sentences/paragraphs using the transformer model from spacy ("en_core_web_trf"). Alternatively we can also use the CPU-optimized model ("en_core_web_lg").

In [ ]:
# Corpus_df_HTML_cleaned_GLLM_v1 for the whole 10-K filings

with open('GLLM/Intermediate_datasets/Corpus_df_HTML_cleaned_GLLM_v1.pkl','rb') as path_name:
    corpus_df = pickle.load(path_name)

# If the model is not already downloaded, you can download it using the following command in your terminal or command prompt:
# python -m spacy download en_core_web_trf
# or you can use the CPU version of the model with the command:
# python -m spacy download en_core_web_lg

# Enable GPU processing (Alternative with CPU model: spacy.require_cpu())
spacy.require_gpu()

# Load the spacy standard transformer model (CPU Alternative: spacy.load("en_core_web_lg"))
nlp = spacy.load('en_core_web_trf',disable=["tagger", "ner", "attribute_ruler", "lemmatizer"])

nlp.max_length = 2500000

# Step 1: Remove missing values in the column "filing_text"
corpus_df = corpus_df.dropna(subset=["filing_text"]).reset_index(drop=True)

# Step 2: Split each filing into sentences

# Use nlp.pipe for batching (NOTE: THIS REQUIRES A GPU/CPU WITH SUFFICIENT MEMORY AND TAKES A COUPLE OF HOURS)
batch_size = 8  # adjust depending on your GPU/CPU memory
all_sentences = []

for doc in tqdm(nlp.pipe(corpus_df["filing_text"], batch_size=batch_size), 
                total=len(corpus_df), desc="Processing filings"):
    all_sentences.append(split_into_sentences(doc))

# Replace the original column with sentence lists
corpus_df["filing_text"] = all_sentences

# Step 3: calculate the number of sentences and share of words with less than 6 words and more than 800 words (efficiency reason for GLLM)
corpus_df["num_sentences_less_than_6_words"] = corpus_df["filing_text"].apply(lambda x: sum(1 for text in x if len(re.findall(r'\b[a-zA-Z]+\b', text)) < 6))
corpus_df["num_sentences_more_than_800_words"] = corpus_df["filing_text"].apply(lambda x: sum(1 for text in x if len(re.findall(r'\b[a-zA-Z]+\b', text)) > 800))

corpus_df["words_in_sentences_less_than_6_words"] = corpus_df["filing_text"].apply(count_words_in_short_sentences)
corpus_df["words_in_sentences_more_than_800_words"] = corpus_df["filing_text"].apply(count_words_in_long_sentences)

corpus_df["share_words_in_sentences_less_than_6_words"] = corpus_df["words_in_sentences_less_than_6_words"] / corpus_df["Word_count"]
corpus_df["share_words_in_sentences_more_than_800_words"] = corpus_df["words_in_sentences_more_than_800_words"] / corpus_df["Word_count"]

print("The average share of words in sentences with less than 6 words is: ",
      corpus_df["share_words_in_sentences_less_than_6_words"].mean())
print("The average share of words in sentences with more than 800 words is: ",
      corpus_df["share_words_in_sentences_more_than_800_words"].mean())

# Step 4: Save the pre-processed corpus as pickle format
corpus_df.to_pickle("GLLM/Intermediate_datasets/Corpus_df_HTML_cleaned_GLLM_v2.pkl")

Next, we perform the last clean-up steps for the GLLM approach by removing sentences for each filing with less than 5 words and more than 800 words, and afterwards dropping obeservations that have less than 500 or more than 5000 remaining sentences. The lower bound assures that at least some content is provided for the analyses and the upper bound caps the interference time and GPU ressources needed via the GLLM.

In [ ]:
with open('GLLM/Intermediate_datasets/Corpus_df_HTML_cleaned_GLLM_v2.pkl','rb') as path_name:
    corpus_df = pickle.load(path_name)

print(f"Starting number of entries {corpus_df.shape[0]}")

# Step 1: drop duplicates based on "filing_key"
corpus_df = corpus_df.drop_duplicates(subset=['filing_key']).reset_index(drop=True)

print(f"Number of entries after dropping duplicates: {corpus_df.shape[0]}")

# Step 2: remove sentences with less than 6 words and more than 800 words (this step makes sure that the GPU memory is not exceeded)
corpus_df["filing_text"] = corpus_df["filing_text"].apply(lambda x: [text for text in x if len(re.findall(r'\b[a-zA-Z]+\b', text)) > 5 and len(re.findall(r'\b[a-zA-Z]+\b', text)) < 800])

# Step 3: count the maximum number of words in a single sentence for each filing
corpus_df["max_words_per_sentence"] = corpus_df["filing_text"].apply(lambda x: max([len(re.findall(r'\b[a-zA-Z]+\b', text)) for text in x]) if x else 0)

# Step 4: count the number of remaining entries ("sentences") for each list in the "filing_text" column
corpus_df["num_entries"] = corpus_df["filing_text"].apply(len)

# Step 5: count the number of words in each entry of the "filing_text" lists and store the counts in the column "Word_count"
corpus_df["Word_count"] = corpus_df["filing_text"].apply(lambda x: sum([len(re.findall(r'\b[a-zA-Z]+\b', text)) for text in x]))

# Step 6: remove rows where num_entries is lower than 500 and larger than 5000 (reason: to filter filings with too less information, and cap the maximum number of entries for gpu efficiency)

# First remove rows with less than 500 entries
corpus_df = corpus_df[corpus_df["num_entries"] >= 500].reset_index(drop=True)
print(f"Number of entries after filtering short filings: {corpus_df.shape[0]}")
# Then remove rows with more than 5000 entries
corpus_df = corpus_df[corpus_df["num_entries"] <= 5000].reset_index(drop=True)
print(f"Number of entries after filtering long filings: {corpus_df.shape[0]}")

# Step 7: Save the cleaned dataframe as pickle format
corpus_df.to_pickle("GLLM/Intermediate_datasets/Corpus_df_HTML_cleaned_GLLM_final.pkl")


<div class="alert-info">
Part 2 (Just for Word-2-Vec)
</div>

To use the 10-K filing text to train and use a Word-2-Vec model, some further pre-processing steps are necessary to ensure the creation of an appropriate and efficient processing pipeline. To be more precise, we use a deep learning model to perform named-entity recognition (NER), replace the named entities with a specified tag (e.g. "Exxon Mobil Corporation" with "[NER_ORG]"), perform dependency parsing, i.e. join words that typically occur together and have one overall meaning (E.g. "financial statement"), and use the gensim package to create further such bigrams/trigrams/quadgrams.

First, we perform NER using the transformer model from spacy ("en_core_web_trf"). Alternatively we can also use the CPU-optimized model ("en_core_web_lg").

We will pass each sentence of the filings separatly into the spacy pipeline.

In [ ]:
# Create new folder to save intermediate files
if not os.path.exists("W2V/Intermediate_datasets"):
    os.makedirs("W2V/Intermediate_datasets")

# Load the dataframe from the pickle file created before

with open('GLLM/Intermediate_datasets/Corpus_df_HTML_cleaned_GLLM_final.pkl','rb') as path_name:
    corpus_df = pickle.load(path_name)

corpus_df.reset_index(inplace=True)

#Initialize a new empty column  
corpus_df["NER_filing_text"] = None

# Enable GPU processing (Alternative with CPU model: spacy.require_cpu())
spacy.require_gpu()

# Load the pre-trained SpaCy transformer model (Alternative: spacy.load("en_core_web_lg"))
nlp = spacy.load('en_core_web_trf',disable=["tagger", "parser", "attribute_ruler", "lemmatizer"])

# Add the 'merge_entities' component to the pipeline
nlp.add_pipe('merge_entities')

# Set the maximum length of the input text to handle large paragraphs
nlp.max_length = 1000000

# Process the text in batches to improve efficiency (NOTE: THIS REQUIRES A GPU/CPU WITH SUFFICIENT MEMORY AND TAKES A COUPLE OF HOURS)
# Time with A40 GPU: ~8.5 hours
# Time with H200 GPU: ~4.3 hours 
# When using CPU model only, the NER task is less precise (F1: 0.86 vs. 0.90).
# Total time can be further reduced by using a smaller model (e.g., en_core_web_sm) 
# or by using parallel processing (e.g., multiprocessing when using multiple CPU cores).

for i in range(0,len(corpus_df["filing_text"])):

    #Let the user know how many documents are pre-processed (in steps of 50) and save iterim results
    if i%50 == 0:
        print(f"{i} documents have been processed")
        corpus_df.to_pickle("W2V/Intermediate_datasets/Corpus_df_W2V_v1_tmp.pkl")

    filing_sentences = pd.DataFrame(corpus_df.loc[i,"filing_text"],columns=["filing_text"])

    # Initialize a list to hold the processed texts
    NER_sentences = []

    if corpus_df.loc[i,"NER_filing_text"] is None:

        for doc in tqdm(nlp.pipe(filing_sentences["filing_text"], batch_size=len(filing_sentences), disable=["tok2vec", "tagger", "parser", "attribute_ruler", "lemmatizer"]), 
                    total=len(filing_sentences), desc=f"Processing filing {i}"):
                
                NER_text = ' '.join([t.text if not t.ent_type_ else '[NER_' + t.ent_type_ + ']' for t in doc])
                NER_sentences.append(NER_text)

        #Free up memory used by the current doc
        torch.cuda.empty_cache()

        corpus_df.at[i,"NER_filing_text"] = NER_sentences
    
    else:
        print(f"Filing {i} already processed. Continue with next filing.")

# Final save
corpus_df.to_pickle("W2V/Intermediate_datasets/Corpus_df_W2V_v1.pkl")

Second, we use the same transformer model as before ("en_core_web_trf") and perform dependency parsing. This is used to detect compound tokens. A compound token is a collection of words that typically occur together and have one overall meaning (E.g. "financial statement"). Alternatively we can also use the CPU-optimized model ("en_core_web_lg").

Before we can perform the dependency parsing, we need to define a function that extracts the dependency relations from the text. The function below takes a text as input and returns a list of tuples, where each tuple contains the dependent and the head of the relation. Finally, the compounds are joined to one word, by replacing the space by "_", and added to the original text.

In [ ]:
ner_tokens = ['NER_CARDINAL', 'NER_DATE', 'NER_EVENT', 'NER_FAC', 'NER_GPE', 
                'NER_LANGUAGE', 'NER_LAW', 'NER_LOC', 'NER_MONEY', 'NER_NORP', 
                'NER_ORDINAL', 'NER_ORG', 'NER_PERCENT', 'NER_PERSON', 
                'NER_PRODUCT', 'NER_QUANTITY', 'NER_TIME', 'NER_WORK_OF_ART',
                'NER_NUMBER']

def dependency_parsing(doc):
    # Create a list of tokens to form the final sentence
    modified_tokens = list(doc)
    compound_head = None
    # Iterate through each token in the sentence
    for token in doc:
        # Check if the token and its "head" are part of a compound noun
        if token.dep_ == 'compound' and modified_tokens[token.i] != None and token != compound_head:
            # Collect all tokens in the compound phrase, including the head
            compound_head = token.head
            compound_tokens = [token]

            for child in compound_head.children:
                if child.dep_ == 'compound' and child not in compound_tokens:
                    compound_tokens.append(child)
            
            if compound_head.dep_ == 'compound':
                compound_tokens.append(compound_head)
                compound_head = compound_head.head
                for child in compound_head.children:
                    if child.dep_ == 'compound' and child not in compound_tokens:
                        compound_tokens.append(child)
            
            # Sort the compound tokens based on their position in the sentence
            compound_tokens.sort(key=lambda t: t.i)
            
            # Merge the compound tokens with their head using underscores
            compound_phrase = "_".join([str(t.text).lower() for t in compound_tokens] + [str(compound_head.text).lower()])
            
            # Replace the head token with the merged phrase
            modified_tokens[compound_head.i] = compound_phrase
            
            # Remove the compound tokens from the list (except the head)
            for t in compound_tokens:
                if t != compound_head:
                    modified_tokens[t.i] = None

    # Filter out the None values from the list of tokens (removed compound parts)
    final_tokens = [token for token in modified_tokens if token is not None]

    # Join the modified tokens back into a sentence and convert to lowercase if not an NER token
    modified_text = " ".join([str(token).lower() if str(token) not in ner_tokens else str(token) for token in final_tokens])

    # Output the modified sentence
    return modified_text

Next, we can load the transformer model and perform dependency parsing sentence by sentence for each filing. 

In [ ]:
with open('W2V/Intermediate_datasets/Corpus_df_W2V_v1.pkl','rb') as path_name:
    corpus_df = pickle.load(path_name)

# Enable GPU processing (Alternative with CPU model: spacy.require_cpu())
spacy.require_cpu()

# Load the pre-trained SpaCy transformer model (Alternative: spacy.load("en_core_web_lg"))en_core_web_trf
nlp = spacy.load("en_core_web_lg")

# Set the maximum length of the input text to handle large documents
nlp.max_length = 1000000

# Process the text in batches to improve efficiency (NOTE: THIS REQUIRES A GPU/CPU WITH SUFFICIENT MEMORY AND TAKES A COUPLE OF HOURS)
# Time with A40 GPU: ~9.5 hours
# Time with H200 GPU: ~4.7 hours 

for i in range(0,len(corpus_df["NER_filing_text"])):

    #Let the user know how many documents are pre-processed (in steps of 50) and save iterim results
    if i%50 == 0:
        print(f"{i} documents have been processed")
        corpus_df.to_pickle("W2V/Intermediate_datasets/Corpus_df_W2V_v2_tmp.pkl")

    filing_sentences = pd.DataFrame(corpus_df.loc[i,"NER_filing_text"],columns=["NER_filing_text"])

    # Initialize a list to hold the processed texts
    dependency_parsing_sentences = []

    # Split the filing in smaller batches to avoid memory issues
    for doc in tqdm(nlp.pipe(filing_sentences["NER_filing_text"], batch_size=len(filing_sentences), disable= ["tagger", "ner", "attribute_ruler", "lemmatizer"]), 
                total=len(filing_sentences), desc=f"Processing filing {i}"):
            
            dependency_parsing_sentences.append(dependency_parsing(doc))

    #Free up memory used by the current doc/s
    
    torch.cuda.empty_cache()

    corpus_df.at[i,"NER_filing_text"] = dependency_parsing_sentences

# Final save
corpus_df.to_pickle("W2V/Intermediate_datasets/Corpus_df_W2V_v2.pkl")


Third, we use the gensim package to create further bigrams/trigrams/quadgrams.

After NER and dependency parsing, we need to prepare the text for the bigram model. Hence, we define some functions to replace remaining numbers (with "[NER_NUMBER]), replace the symbole "-" with "_" (e.g. activity-based to activity_based), remove spaces before closing-brackets and after opening-brackets, etc... This helps to decrease the vocabulary size further. For example, we do not want to have separate embeddings for different numbers of special symboles.

In [ ]:
# First, we define some functions to clean the text and prepare it for the Word2Vec model. 
# These function will be applied at different stages of the pre-processing pipeline, depending on the specific requirements of the analysis.

def replace_symboles(text):
    """
    Function to replace - by  "_"
    """
    text = re.sub(r' \- ', '_',text)
    text = re.sub(r'\- ', '_',text)
    text = re.sub(r' \-', '_',text)
    return text

def remove_linebreaks(text):
    """
    Function to remove line breaks
    """
    text = re.sub(r'\n{1,}', ' ',text)
    text = re.sub(r'\s{2,}', ' ', text.strip())

    return text


def replace_numbers(text):
    """
    Function to replace numbers by the tag "[NER_NUMBER]"
    """
    text = re.sub(r'\b\d+(?:[.,]\s*\d{3})*(?:[.,]\s*\d+)?[a-zA-Z]?\b', '[NER_NUMBER]',text)
    return text

def replace_brackets_space(text):
    """
    Function to replace numbers by the symbole "#"
    """
    text = re.sub(r'\[ ', '[',text)
    text = re.sub(r' \]', ']',text)
    return text


def remove_nonalpha(text):
    """
    Function to remove non-alpha characters except for underscores and brackets.
    """
    text = re.sub(r"[^A-Za-z\_\[\]]+", ' ', str(text))
    text = re.sub(r'\s{2,}', ' ', text.strip())
    return text

def cleaning(row, stop_words):
    # Removes stopwords
    doc = [token for token in row.split() if token not in stop_words]
    # Word2Vec uses context words to learn the vector representation of a target word,
    # if a sentence is only one to three words long the benefit for the training is very small
    if len(re.findall(r'\b(\w+\_*)+\b',str(' '.join(doc)))) > 3:
        return ' '.join(doc)
    
def replace_multiple_underlines(text):
    """
    Function to replace multiple underscores with a single underscore.
    """
    text = re.sub(r"(\_){2,}", '_', str(text))
    return text

def remove_underline_space(text):
    """
    Function to remove remaining underscores which do not connect words.
    """
    text = re.sub(r' \_ ', ' ', str(text))
    text = re.sub(r'\_ ', ' ', str(text))
    text = re.sub(r' \_', ' ', str(text))
    text = re.sub(r'\s{2,}', ' ', text.strip())
    return text

def concanate_letter_s(text):
    text = re.sub(r' s ', '_s ', str(text))
    text = re.sub(r'\s{2,}', ' ', text.strip())
    return text

def remove_single_letters(text):
    text = re.sub(r'\b([b-hj-z])\b', ' ', str(text))
    text = re.sub(r'\s{2,}', ' ', text.strip())
    return text

#Next, we define a function (which splits the sentences into a list of words --> expected input to train the ngram model; 
# and remove single letter words and too long words, as we want to avoid that some remaining single letters ("noise") are joined and the created ngrams are too long)
def custom_simple_preprocess(doc, min_len=2, max_len=20):
    """
    Convert a document into a list of tokens.
    This function is a modified version of gensim's simple_preprocess to avoid lowercasing.
    """
    tokens = [token for token in doc.split() if min_len <= len(token) <= max_len]
    return tokens

# Helper class which helps to stream 10-K filing sentences (memory efficient) from disk into Gensim  
class CleanFilings:
    """An iterator that yields tokenized sentences (lists of str) from a csv file."""

    # Initialize with the path to the corpus file
    def __init__(self, corpus_path):
        self.corpus_path = corpus_path
    
    # Define the iterator
    def __iter__(self):
        
        chunk_size = 50000  # Adjust chunk size as needed
        
        # Stream the file in chunks
        for chunk in pd.read_csv(self.corpus_path, chunksize=chunk_size):
            for sentence in chunk['NER_filing_text']:
                yield custom_simple_preprocess(sentence) # Yield the sentence as a list of tokens

In [ ]:
# Now we load the corpus dataframe and apply the cleaning functions to the "NER_filing_text" column.
with open('W2V/Intermediate_datasets/Corpus_df_W2V_v2.pkl','rb') as path_name:
    corpus_df = pickle.load(path_name)

# Replace symboles etc....
corpus_df["NER_filing_text"] = corpus_df["NER_filing_text"].apply(lambda x:list(map(remove_linebreaks, x)))

corpus_df["NER_filing_text"] = corpus_df["NER_filing_text"].apply(lambda x:list(map(replace_symboles, x)))

corpus_df["NER_filing_text"] = corpus_df["NER_filing_text"].apply(lambda x:list(map(replace_numbers, x)))

corpus_df["NER_filing_text"] = corpus_df["NER_filing_text"].apply(lambda x:list(map(replace_brackets_space, x)))

# Final save
corpus_df.to_pickle("W2V/Intermediate_datasets/Corpus_df_W2V_v3.pkl")


Further, we remove some words and symboles that we do not want to join as ngrams.

In [ ]:
# download nltk stopwords, if not yet done
# import nltk
# nltk.download('stopwords')

#Import the stopword list from the NLTK package
from nltk.corpus import stopwords
stop_words = stopwords.words("english")

# Add words to the stop_words list, i.e. words that are deleted before phrase model is trained
# NER tokens are added to the stopword list, as they should not be part of the phrase model 
# (otherwise, the phrase model would create phrases with NER tokens, which is not desired in the context of this analysis)
additional_stop_words = ['[NER_CARDINAL]', '[NER_DATE]', '[NER_EVENT]', '[NER_FAC]', '[NER_GPE]', 
                          '[NER_LANGUAGE]', '[NER_LAW]', '[NER_LOC]', '[NER_MONEY]', '[NER_NORP]', 
                          '[NER_ORDINAL]', '[NER_ORG]', '[NER_PERCENT]', '[NER_PERSON]', 
                          '[NER_PRODUCT]', '[NER_QUANTITY]', '[NER_TIME]', '[NER_WORK_OF_ART]',
                          '[NER_NUMBER]'] 

# Extend the stop_words list with additional_stop_words
stop_words.extend(additional_stop_words)

# Some stopwords should not be removed. Hence, we remove some specified words from the stop_words list
words_to_remove = {'it', 'of', 'in', 'on', 'at'}

stop_words = [word for word in stop_words if word not in words_to_remove]

In [ ]:
# Pass the sentences three times through the bigram model in order to capture bi-, tri-, and quadgrams.

corpus_paths = ["W2V/Intermediate_datasets/Corpus_df_W2V_v3", "W2V/Intermediate_datasets/Corpus_df_W2V_v3_w_bigrams", "W2V/Intermediate_datasets/Corpus_df_W2V_v3_w_trigrams"
                ]

for corpus_path in corpus_paths:
    with open(f"{corpus_path}.pkl",'rb') as path_name:
        corpus_df = pickle.load(path_name)

    filing_sentences = list(corpus_df["NER_filing_text"])

    del corpus_df

    # Part 1: Prepare the sentences for the phrase model training
    print(f"Preparing the sentences of {corpus_path} for the phrase model training...")
    # Create a list of all sentences in the corpus
    filing_sentences = pd.DataFrame(itertools.chain(*filing_sentences), columns=["NER_filing_text"])
    # Remove any remaining non-alpha characters
    filing_sentences["NER_filing_text"] = filing_sentences["NER_filing_text"].apply(lambda x:remove_nonalpha(x))
    # Remove the words that have been defined above and remove sentences that are only one to three words long. 
    # Word2Vec uses context words to learn the vector representation of a target word, benefit for the training is very small
    filing_sentences["NER_filing_text"] = filing_sentences["NER_filing_text"].apply(lambda x:cleaning(x,stop_words=stop_words))
    # Remove any duplicate sentence
    filing_sentences = filing_sentences['NER_filing_text'].dropna().drop_duplicates()
    # Save the cleaned sentences to stream them into the phrase model training
    filing_sentences.to_csv(f"{corpus_path}_cleaned_sentences.csv", index=False)
    del filing_sentences

    # Part 2: Train the phrase model
    # Now, we can start training the ngram model. In order to detect ngrams we use the the default scoring function from gensim. 
    # The minimum count is set to to 10, i.e., just ngrams that occur at a minimum of 10 times will be considered, 
    # the threshold of the scoring function is set to 0.1 (is quite low, but management accounting practices occur just a few time and 
    # setting the threshold higher might lead to not detecting them). 
    
    corpus_path = f"{corpus_path}_cleaned_sentences.csv"
    sentences = CleanFilings(corpus_path)

    print(f"Training the bigram model on {corpus_path}...")

    # Train a bigram model (min_count = 10 if whole report is considered)
    bigram_model = Phrases(sentences=sentences ,min_count=10 , threshold=0.1,max_vocab_size=40000000,scoring='default', connector_words=ENGLISH_CONNECTOR_WORDS) 

    # Part 3: Save and apply the trained phrase model
    # Convert the bigram model to a Phraser object
    bigram_phraser = Phraser(bigram_model)

    # Save the trained bigram model to disk for later use
    bigram_phraser.save(f"{corpus_path}_bigram_phraser.pkl")

    # Now, we can apply the phraser on our corpus of 10-k filings (We use the corpus in which each filing has its own list of 10-k sentences).
    with open(f"{corpus_path}.pkl",'rb') as path_name:
        corpus_df = pickle.load(path_name)

    print(f"Applying the bigram model to {corpus_path}...")
    
    # Apply the bigram model to each sentence in the "NER_filing_text" column
    corpus_df["NER_filing_text"] = corpus_df["NER_filing_text"].apply(lambda x: [' '.join(bigram_model[sentence.split()]) for sentence in x if sentence is not None])

    # Save the updated dataframe
    if corpus_path == "W2V/Intermediate_datasets/Corpus_df_W2V_v3":
        corpus_df.to_pickle("W2V/Intermediate_datasets/Corpus_df_W2V_v3_w_bigrams.pkl") 
    elif corpus_path == "W2V/Intermediate_datasets/Corpus_df_W2V_v3_w_bigrams": 
        corpus_df.to_pickle("W2V/Intermediate_datasets/Corpus_df_W2V_v3_w_trigrams.pkl")
    else:
        corpus_df.to_pickle("W2V/Intermediate_datasets/Corpus_df_W2V_v3_final.pkl")

    del corpus_df

del bigram_counter, most_common_bigrams, bigram_model, bigram_phraser, sentences


<div class="alert-info">
Step 2.5: Train the Word-2-Vec model
</div>

Before we can start to train the W2V model, a final clean-up of the filing sentences is done. This step is important since the output of the phraser model still includes, e.g., symboles that should not be considered in the W2V training. For exmaple: any non-alphanumeric symbole should not be considered etc... 

In [ ]:
with open('W2V/Intermediate_datasets/Corpus_df_W2V_v3_final.pkl','rb') as path_name: 
    corpus_df = pickle.load(path_name)

#First, we remove all non-alpha characters from the text
corpus_df["NER_filing_text"] = corpus_df["NER_filing_text"].apply(lambda x:list(map(remove_nonalpha, x)))
#Second, we replace multiple underlines in words by a single underline (e.g. "word__word" -> "word_word")
corpus_df["NER_filing_text"] = corpus_df["NER_filing_text"].apply(lambda x:list(map(replace_multiple_underlines, x)))
#Third, we remove all underlines that are followed or preceded by a space (e.g. " _word" or "word_ " -> "word")
corpus_df["NER_filing_text"] = corpus_df["NER_filing_text"].apply(lambda x:list(map(remove_underline_space, x)))
#Fourth, we concanate the letter "s" to the previous word if it is separated by a space (e.g. "word s" -> "word_s")
corpus_df["NER_filing_text"] = corpus_df["NER_filing_text"].apply(lambda x:list(map(concanate_letter_s, x)))
#Fifth, we remove all single letters from the text (e.g. "a" or "b" -> " ")
corpus_df["NER_filing_text"] = corpus_df["NER_filing_text"].apply(lambda x:list(map(remove_single_letters, x)))

#Final save
corpus_df.to_pickle("W2V/Intermediate_datasets/Corpus_df_W2V_v4.pkl")

del corpus_df

Afterwards, we create the training dataset for the W2V model. This includes creating one list that stores all sentences in the 10-k filings and dropping duplicates (to train the W2V model it is not important in which filings the sentence occures, but that we have all kind of sentences that can occure in these reports). Note that we will train the W2V model for the corpus with trigrams and quadgrams generated by the gensim ngram model separatly. This is done to compare the resulting word vectors and to compare the resulting synonyms for our own MAP dictionary (see later).

In [ ]:

with open('W2V/Intermediate_datasets/Corpus_df_W2V_v4.pkl','rb') as path_name: 
    corpus_df = pickle.load(path_name)

filing_sentences = list(corpus_df["NER_filing_text"])

del corpus_df

# Creating a list of all sentences in the corpus
filing_sentences = pd.DataFrame(itertools.chain(*filing_sentences), columns=["NER_filing_text"])
# Dropping any duplicate sentences
filing_sentences = filing_sentences['NER_filing_text'].dropna().drop_duplicates()

# Saving the cleaned sentences to a CSV file
filing_sentences.to_csv('W2V/Intermediate_datasets/Corpus_df_W2V_cleaned_sentences_final.csv', index=False) 

del filing_sentences

First, we define a class that helps us to stream the training dataset from disk.

In [ ]:
# Helper class to input 10-K filing sentences from our data frame into Gensim
# Note: It replaces the previous CleanFilings class, as the cleaning is slightly different (e.g., we do not use short/long word filtering etc.)

class CleanFilings:
    """An iterator that yields tokenized sentences (lists of str) from a pickle file."""

    def __init__(self, corpus_path):
        self.corpus_path = corpus_path

    def __iter__(self):
 
        chunk_size = 50000  # Adjust chunk size as needed
        
        # Stream the file in chunks
        for chunk in pd.read_csv(self.corpus_path, chunksize=chunk_size):
            for sentence in chunk['NER_filing_text']:
                if len(re.findall(r'\b\w+\_*\w*\b',str(sentence))) < 4: # Just process sentences with a minimum of 4 words
                    pass
                else:
                    yield sentence.split() 

Next, we determine the number of CPUs that are available for training and perform a check whether the "Fast Version" of the Word2Vec Model is available (The Fast Version uses parallel processing to train the word vectors, which is a lot fast --> Expected output: FAST_VERSION = 0 or 1).

In [ ]:
import multiprocessing

cores = multiprocessing.cpu_count() # Count the number of cores in a computer

print(f"Number of CPU cores available: {cores}")

print(f"FAST_VERSION: {word2vec.FAST_VERSION}")


Now, we can initialize the Word2Vec model, build up the vocab, and start training (We split-up the W2V steps to better track them individually).

First, we initialize the Word2Vec model.

In [ ]:
# CAUTION: Running the code might take a while
from time import time  # To time our operations
import logging  # Setting up the loggings to monitor gensim

# Enable logging at the INFO level
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

# Set the path to the cleaned sentences CSV file
corpus_path = 'W2V/Intermediate_datasets/Corpus_df_W2V_cleaned_sentences_final.csv'  # path to the cleaned sentences CSV file
filing_text = CleanFilings(corpus_path)

# Define training parameters for the Word2Vec model
emb_dim = 300  # Size of the embedding vector
min_count = 10  # Minimum frequency count of words to be included in the model
window_size = 5  # Size of the context window

# Initialize the Word2Vec model using the specified parameters
model = Word2Vec(min_count=min_count,  
                 vector_size=emb_dim, 
                 workers=cores-1)    # for parallel computing

Second, we build the vocabulary of the model.

In [ ]:
t = time()

# Start building the vocabulary from the cleaned sentences
model.build_vocab(filing_text, progress_per=100000)

print('Time to build vocab: {} mins'.format(round((time() - t) / 60, 2)))

Third, we train the Word2Vec model.

In [ ]:
t = time()

# Start training the Word2Vec model on the cleaned sentences
model.train(filing_text, total_examples=model.corpus_count, epochs=20, report_delay=5)

print('Time to train the model: {} mins'.format(round((time() - t) / 60, 2)))

# Save trained word vectors to disk
file="W2V/Dictionary_creation/w2v_filing_text.model" 

# Set binary to True to save disk space; false facilitates inspecting the embeddings in a text editor
save_as_bin = True

# Save the word vectors in the word2vec format
model.wv.save_word2vec_format(file, binary=save_as_bin) 

Similarly, we can load the word vectors with the following code:

In [ ]:
from gensim.models import KeyedVectors

file="W2V/Dictionary_creation/w2v_filing_text.model" 

# Load model from disk
w2v = KeyedVectors.load_word2vec_format(file, binary=True)

# Then, you can use the trained Word2Vec model to find the most similar words to a given word. For example, to find the top 20 words most similar to "cost_management", you can use the following code:
w2v.most_similar(positive=['cost_management'], topn=20)